In [ ]:
#| default_exp project

# Python project helpers

> Format-preserving TOML edits, project and package discovery, and version reading, writing, and bumping.

Utilities shared by Python project tooling (`nbdev`, `fastship`, and friends): lightweight format-preserving TOML edits, walking up to project markers and locating package directories, and reading, writing, and bumping `X.Y.Z` / `X.Y.Z.postN` versions.

In [ ]:
from fastcore.utils import *
from fastcore.pyparse import parse_py
import pprint

In [ ]:
from fastcore.test import *
from tempfile import TemporaryDirectory

## Format-preserving TOML editing

In [ ]:
def fmt_toml_val(
    v # Scalar or sequence of strings to format
)->str:
    "Format a simple Python scalar or string list as a TOML value literal"
    if isinstance(v, bool): return str(v).lower()
    if isinstance(v, (list, tuple)): return '[' + ', '.join(f'"{o}"' for o in v) + ']'
    return f'"{v}"'

def toml_section_bounds(
    lines:list[str], # Lines of a TOML file (without trailing newlines)
    section:str, # Dotted section header name without brackets, e.g. `'project.urls'`
)->tuple[int|None, int|None]: # `(start_idx, end_idx)` or `(None, None)` when absent
    "Find the header index and exclusive end index of `[section]` in `lines`"
    hdr = f'[{section}]'
    start = next((i for i,ln in enumerate(lines) if ln.strip() == hdr), None)
    if start is None: return None, None
    end = next((i for i in range(start + 1, len(lines)) if re.match(r'^\s*\[.*\]\s*$', lines[i])), len(lines))
    return start, end

`fmt_toml_val` formats simple inline literals and `toml_section_bounds` locates `[section]` line ranges without re-serializing the file:

In [ ]:
test_eq([fmt_toml_val(True), fmt_toml_val(['a', 'b']), fmt_toml_val('hi')], ['true', '["a", "b"]', '"hi"'])
test_eq(toml_section_bounds(['[a]', 'x = 1', '', '[b]', 'y = 2'], 'a'), (0, 3))
test_eq(toml_section_bounds(['[a]', 'x = 1'], 'missing'), (None, None))

In [ ]:
def replace_toml_section_key(
    p:str|Path, # TOML file to update in place
    section:str, # Section header name without brackets
    key:str, # Key name to replace inside `[section]`
    val:str, # New string value
):
    "Replace `key = ...` inside `[section]` of `p`, preserving surrounding formatting"
    p = Path(p)
    lines = p.read_text(encoding='utf-8').splitlines()
    start,end = toml_section_bounds(lines, section)
    pat = re.compile(rf'^\s*{re.escape(key)}\s*=')
    hits = [i for i in range(start + 1, end) if pat.match(lines[i])] if start is not None else []
    if not hits: raise ValueError(f'Could not find {key!r} in [{section}] of {p}')
    lines[hits[0]] = f'{key} = "{val}"'
    p.write_text('\n'.join(lines) + '\n', encoding='utf-8')

`replace_toml_section_key` rewrites a single string key in place and raises `ValueError` if missing:

In [ ]:
_td = TemporaryDirectory()
_toml = Path(_td.name)/'pyproject.toml'
_toml.write_text('[project]\nname = "demo"\nversion = "0.1.0"\n\n[project.urls]\nRepository = "https://github.com/org/demo"\n\n[project.scripts]\na = "b:c"\n')
replace_toml_section_key(_toml, 'project', 'version', '0.2.0')
assert 'version = "0.2.0"' in _toml.read_text()
with expect_fail(ValueError): replace_toml_section_key(_toml, 'project', 'missing', '1')

In [ ]:
def ensure_toml_section(
    p:str|Path, # TOML file to update in place
    section:str, # Section header name without brackets
    items:dict, # Key/value pairs to ensure exist in `[section]`
    replace:bool=False, # Overwrite existing matching keys (case-insensitive)?
):
    "Ensure `[section]` in `p` contains `items`, preserving existing keys and blank-line spacing"
    p = Path(p)
    lines = p.read_text(encoding='utf-8').splitlines()
    start,end = toml_section_bounds(lines, section)
    vals = {k: f'{k} = {fmt_toml_val(v)}' for k,v in items.items()}
    if start is None:
        if lines and lines[-1].strip(): lines.append('')
        lines += [f'[{section}]', *vals.values()]
    else:
        existing = {m.group(1).lower(): i for i in range(start + 1, end) if (m := re.match(r'^\s*([A-Za-z0-9_-]+)\s*=', lines[i]))}
        if replace:
            for k,val in vals.items():
                if k.lower() in existing: lines[existing[k.lower()]] = val
        ins = end
        while ins > start + 1 and not lines[ins - 1].strip(): ins -= 1
        lines[ins:ins] = [val for k,val in vals.items() if k.lower() not in existing]
    p.write_text('\n'.join(lines) + '\n', encoding='utf-8')

`ensure_toml_section` inserts missing keys before any trailing blank line, matches existing keys case-insensitively, and appends missing sections:

In [ ]:
ensure_toml_section(_toml, 'project.urls', {'repository': 'https://other/x', 'Documentation': 'https://org.github.io/demo/'})
ensure_toml_section(_toml, 'tool.demo', {'flag': True, 'items': ['a', 'b']})
test_eq(_toml.read_text(), '[project]\nname = "demo"\nversion = "0.2.0"\n\n[project.urls]\nRepository = "https://github.com/org/demo"\nDocumentation = "https://org.github.io/demo/"\n\n[project.scripts]\na = "b:c"\n\n[tool.demo]\nflag = true\nitems = ["a", "b"]\n')
_td.cleanup()

## Project and package discovery

In [ ]:
def find_marker(
    start:str|Path|None, # Starting directory or file (defaults to current working directory)
    *fnames:str, # Marker filenames to look for in priority order at each directory level
)->Path:
    "Find the nearest of `fnames` in `start` or its parent directories"
    p = Path(start).expanduser().resolve() if start else Path().resolve()
    if p.is_file(): p = p.parent
    for d in (p, *p.parents):
        for f in fnames:
            if (d/f).exists(): return d/f
    raise FileNotFoundError(f"Could not find {' or '.join(fnames)} (searched parents from {p})")

def find_pyproject(
    start:str|Path|None=None, # Starting directory or file (defaults to current working directory)
)->Path:
    "Find the nearest `pyproject.toml` in `start` or its parent directories"
    return find_marker(start, 'pyproject.toml')

def is_uv_workspace(data:dict)->bool:
    "True when parsed `pyproject.toml` dict `data` defines `[tool.uv.workspace]`"
    return nested_idx(data, 'tool', 'uv', 'workspace') is not None

`find_marker` and `find_pyproject` walk parent directories to locate project files, and `is_uv_workspace` detects uv workspace roots:

In [ ]:
_ws_td = TemporaryDirectory()
_ws = Path(_ws_td.name)
(_ws/'pyproject.toml').write_text('[tool.uv.workspace]\nmembers = ["*"]\n')
_proj = _ws/'my-pkg'
(_proj/'python'/'my_pkg').mkdir(parents=True)
(_proj/'python'/'my_pkg'/'__init__.py').write_text('')
(_proj/'pyproject.toml').write_text('[project]\nname = "my-pkg"\n[tool.maturin]\npython-source = "python"\n')
test_eq(find_pyproject(_proj/'python'), (_proj/'pyproject.toml').resolve())
assert is_uv_workspace((_ws/'pyproject.toml').read_toml())

In [ ]:
def project_root(
    path:str|Path|None=None, # Starting path (defaults to current working directory)
)->tuple[Path, dict, Path|None]: # `(root, data, pyproject_path)`
    "Nearest non-workspace project `(root, data, pyproject_path)`, falling back to `(resolved_path, {}, None)`"
    try:
        pyproj = find_pyproject(path)
        if not is_uv_workspace(data := pyproj.read_toml()): return pyproj.parent.resolve(), data, pyproj
    except FileNotFoundError: pass
    return Path(path or '.').expanduser().resolve(), {}, None

`project_root` skips uv workspace roots so member directories without a `pyproject.toml` (such as pure Rust crates) resolve cleanly to themselves:

In [ ]:
_root,_data,_pp = project_root(_proj/'python')
test_eq((_root, _pp), (_proj.resolve(), (_proj/'pyproject.toml').resolve()))
_crate = _ws/'pure-crate'
_crate.mkdir()
test_eq(project_root(_crate), (_crate.resolve(), {}, None))

In [ ]:
def norm_mod(name:str)->str:
    "Normalize a project name to a Python import/package name"
    return re.sub(r'[^0-9a-zA-Z_]+', '_', name.strip().replace('-', '_'))

def _pkg_bases(root:Path, data:dict)->list[Path]:
    bases = []
    if mat_src := nested_idx(data, 'tool', 'maturin', 'python-source'): bases.append(root/mat_src)
    if pkg_dir := nested_idx(data, 'tool', 'setuptools', 'package-dir', ''): bases.append(root/pkg_dir)
    pkgs = nested_idx(data, 'tool', 'setuptools', 'packages')
    if isinstance(pkgs, dict): bases.extend(root/w for w in L(nested_idx(pkgs, 'find', 'where') or []))
    return list(dict.fromkeys([*bases, root/'src', root]))

`norm_mod` normalizes a distribution name to a Python module identifier:

In [ ]:
test_eq(norm_mod('my-pkg'), 'my_pkg')

In [ ]:
def find_pkg_path(
    root:str|Path, # Project root directory
    data:dict=None, # Parsed `pyproject.toml` dict (read from `root/pyproject.toml` if None)
    pkg:str=None, # Explicit package directory name override
)->Path:
    "Locate the Python package directory under `root` from `pyproject.toml` config or standard layouts"
    root = Path(root)
    if data is None: data = (root/'pyproject.toml').read_toml() if (root/'pyproject.toml').exists() else {}
    if (lp := nested_idx(data, 'tool', 'nbdev', 'lib_path')) and (root/lp/'__init__.py').exists(): return root/lp
    bases = [o for o in _pkg_bases(root, data) if o.exists()]
    def _top(name): return (b/name for b in bases if (b/name/'__init__.py').exists())
    if pkg:
        nested = (p.parent for b in bases for p in b.rglob('__init__.py') if p.parent.name == pkg)
        if found := next(_top(pkg), None) or next(nested, None): return found
        raise FileNotFoundError(f'Could not find {pkg}/__init__.py under {root}')
    if (nm := nested_idx(data, 'project', 'name')) and (found := next(_top(norm_mod(nm)), None)): return found
    if not is_uv_workspace(data):
        if found := next((p for b in bases for p in b.iterdir() if p.is_dir() and (p/'__init__.py').exists() and not p.name.startswith('.')), None):
            return found
    raise FileNotFoundError(f'Could not find package directory under {root}')

`find_pkg_path` checks `[tool.nbdev].lib_path`, maturin/setuptools source roots, explicit `pkg` overrides, normalized `[project].name`, and standard layouts:

In [ ]:
test_eq(find_pkg_path(_root, _data), _root/'python'/'my_pkg')
test_eq(norm_mod('my-pkg'), 'my_pkg')
with expect_fail(FileNotFoundError): find_pkg_path(_crate, {})
_ws_td.cleanup()

## Version reading, writing, and bumping

In [ ]:
_re_ver_any = re.compile(r'^__version__\s*=.*$', re.MULTILINE)
_re_ver_val = re.compile(r'^__version__\s*=\s*[\'"]([^\'"]+)[\'"]\s*$', re.MULTILINE)

def _init_path(path:str|Path)->Path:
    p = Path(path)
    return p if p.suffix == '.py' else p/'__init__.py'

def read_version(
    path:str|Path, # Package directory or `__init__.py` file
)->str|None:
    "Read literal `__version__` from `path` (or `path/__init__.py`), returning `None` if absent"
    p = _init_path(path)
    if not p.exists(): return None
    m = _re_ver_val.search(p.read_text(encoding='utf-8'))
    return m.group(1) if m else None

`read_version` extracts a literal `__version__` string from a package directory or `.py` file, returning `None` when missing:

In [ ]:
test_is(read_version(Path('nonexistent_pkg')), None)

In [ ]:
def set_version(
    path:str|Path, # Package directory or `__init__.py` file
    version:str, # New version string
)->bool: # True if file was created or updated
    "Set literal `__version__` in `path` (or `path/__init__.py`), inserting after shebang/coding/docstring when absent"
    p = _init_path(path)
    ver_line = f'__version__ = "{version}"'
    code = p.read_text(encoding='utf-8') if p.exists() else ''
    if _re_ver_any.search(code): new = _re_ver_any.sub(ver_line, code, count=1)
    else:
        lines = code.splitlines()
        ins = 1 if lines and lines[0].startswith('#!') else 0
        for i in range(ins, min(ins + 2, len(lines))):
            if re.match(r'^#.*coding[:=]\s*[-\w.]+', lines[i]): ins = i + 1
        if (m := parse_py(code, safe=True)) and m.doc_node: ins = max(ins, m.doc_node.end_lineno)
        tail = lines[ins:]
        if tail and tail[0].strip(): tail = ['', *tail]
        new = '\n'.join([*lines[:ins], ver_line, *tail]) + '\n'
    return write_if_changed(p, new)

`read_version` and `set_version` accept either a package directory or a `.py` file; `set_version` updates an existing `__version__` in place without adding extra blank lines, or inserts it after any module docstring:

In [ ]:
with TemporaryDirectory() as td:
    pkg = Path(td)/'demo'
    init = pkg/'__init__.py'
    init.mk_write('"Demo pkg."\n\nimport os\n')
    test_is(read_version(pkg), None)
    assert set_version(pkg, '0.1.0')
    test_eq(read_version(pkg), '0.1.0')
    test_eq(init.read_text(), '"Demo pkg."\n__version__ = "0.1.0"\n\nimport os\n')
    assert not set_version(init, '0.1.0') and set_version(init, '0.1.1')
    test_eq(init.read_text(), '"Demo pkg."\n__version__ = "0.1.1"\n\nimport os\n')

In [ ]:
def bump_version(
    v:str, # Current version in `X.Y.Z` or `X.Y.Z.postN` format
    part:int=None, # Component to bump (`0`=major, `1`=minor, `2`=patch); default bumps `.postN` if present else patch
    unbump:bool=False, # Decrement instead of incrementing (clamped at 0)?
)->str:
    "Bump `.postN` by default when present, else semver `part` (0=major, 1=minor, 2=patch)"
    m = re.fullmatch(r'(\d+)\.(\d+)\.(\d+)(?:\.post(\d+))?', v or '0.0.0')
    if not m: raise ValueError(f'Invalid version {v!r}: expected X.Y.Z or X.Y.Z.postN')
    delta = -1 if unbump else 1
    if part is None and m[4] is not None: return f'{m[1]}.{m[2]}.{m[3]}.post{max(0, int(m[4]) + delta)}'
    parts = [int(m[1]), int(m[2]), int(m[3])]
    p = 2 if part is None else part
    if not 0 <= p < 3: raise ValueError(f'part must be 0, 1, or 2 (got {p})')
    parts[p] = max(0, parts[p] + delta)
    for i in range(p + 1, 3): parts[i] = 0
    return '.'.join(map(str, parts))

`bump_version` accepts strict `X.Y.Z` and `X.Y.Z.postN` versions, defaulting to `.postN` when present or patch otherwise:

In [ ]:
test_eq([bump_version('1.2.3'), bump_version('1.2.3', 1), bump_version('1.2.3', 0)], ['1.2.4', '1.3.0', '2.0.0'])
test_eq(bump_version('0.0.0', unbump=True), '0.0.0')
test_eq([bump_version('2026.10.28.post0'), bump_version('2026.10.28.post2', unbump=True), bump_version('2026.10.28.post2', 2)],
    ['2026.10.28.post1', '2026.10.28.post1', '2026.10.29'])
with expect_fail(ValueError): bump_version('9.7.1.2')
with expect_fail(ValueError): bump_version('1.2.3', 3)

## Generated Python module variables

In [ ]:
def set_py_var(
    path:str|Path, # Python file to create or update
    name:str, # Top-level variable name to set or remove
    val, # Value to format with `pprint.pformat`, or `None` to remove
    *, header:str='# AUTOGENERATED! DO NOT EDIT!\n', # Header prepended when creating a missing file
    width:int=140, # Line width passed to `pprint.pformat`
    indent:int=2, # Indentation passed to `pprint.pformat`
    compact:bool=False, # Compact formatting passed to `pprint.pformat`
)->bool: # True if file was created, updated, or unlinked
    "Create, update, or remove a top-level `name = ...` assignment in `path` via AST"
    p = Path(path)
    code = p.read_text(encoding='utf-8') if p.exists() else ''
    lines = code.splitlines()
    m = parse_py(code, safe=True) if code else None
    n = m.assigns.get(name) if m else None
    if val is None:
        if not n: return False
        del lines[n.lineno - 1:n.end_lineno]
        if not any(ln.strip() and not ln.lstrip().startswith('#') for ln in lines) and '# autogenerated' in code.lower():
            p.unlink()
            return True
        txt = re.sub(r'\n{3,}', '\n\n', '\n'.join(lines)).strip('\n')
        return write_if_changed(p, (txt + '\n') if txt else '')
    stmt = f'{name} = {pprint.pformat(val, width=width, indent=indent, compact=compact, sort_dicts=False)}'
    if not p.exists(): return write_if_changed(p, ((header.rstrip('\n') + '\n\n') if header else '') + stmt + '\n')
    if n: lines[n.lineno - 1:n.end_lineno] = stmt.splitlines()
    else: lines += ([''] if lines and lines[-1].strip() else []) + stmt.splitlines()
    return write_if_changed(p, '\n'.join(lines) + '\n')


`set_py_var` creates, updates, and removes top-level assignments in place while preserving sibling variables, unlinking autogenerated files when their last variable is removed:

In [ ]:
with TemporaryDirectory() as td:
    p = Path(td)/'_meta.py'
    assert set_py_var(p, 'a', {'k': 1}) and not set_py_var(p, 'a', {'k': 1})
    assert set_py_var(p, 'b', [1, 2]) and set_py_var(p, 'a', {'k': 2})
    test_eq(parse_py(p.read_text()).literals, {'a': {'k': 2}, 'b': [1, 2]})
    assert set_py_var(p, 'a', None) and parse_py(p.read_text()).literals == {'b': [1, 2]}
    assert set_py_var(p, 'b', None) and not p.exists() and not set_py_var(p, 'b', None)